# 🤖 Build Your Own RAG Chatbot from Scratch


## A Beginner-Friendly Guide (Zero to Working Chatbot)

    Welcome! Today, you are going to build a chatbot that doesn't just "guess" answers, 
    but actually reads documents to give you facts.

## 📚 What is a "Normal" Chatbot?

    A normal AI (like ChatGPT) is like a student who memorized a million books years ago 
    but isn't allowed to look at new notes. 
    If you ask it about something that happened yesterday or a private document you wrote, 
    it will hallucinate (confidently lie).

## 🔍 What is a RAG Chatbot?

    RAG stands for Retrieval-Augmented Generation.
    Think of it as giving the AI an "Open Book Exam." 
        1. Retrieval: The AI looks through your specific documents to find the right page.
        2. Augmented: It adds that information to your question.
        3. Generation: It writes a perfect answer based only on those facts.

### Flowchart of RAG

```mermaid
flowchart TD
    Q[User Question] --> EmbedQ[Embed];

    DB[Vector DB] -->|top-k| Retrieve[Retrieve relevant chunks];

    EmbedQ --> Retrieve;
    Retrieve --> Context[Augmented Prompt = query + chunks];

    Context --> LLM(LLM);
    LLM -->|generates| R[Answer];

In [ ]:
# STEP 1: INSTALLING OUR TOOLS
# We need to download some 'libraries' (pre-written code) to help us.

# 'langchain' is the main framework for building AI apps
!pip install langchain langchain-community langchain-huggingface 

# 'chromadb' is our "Vector Database" (the AI's library where it stores facts)
!pip install chromadb 

# 'sentence-transformers' helps the AI understand the "meaning" of sentences
!pip install sentence-transformers 

# 'pypdf' lets the AI read PDF files
!pip install pypdf 

# 'gradio' creates a pretty chat window for us to type in
!pip install gradio

!pip install ollama

!pip install PyMuPDF
print("✅ All tools installed successfully!")

## 🧪 Experiment 1: The "Normal" AI (No Context)

In [1]:
# 1. Import the library we need
import ollama   # This tool lets Python talk to your local AI models (Ollama)

# 2. Write the exact question you want to ask the AI
question = """
For the F/A-18 Hornet, below what aircraft gross weight does the Flight Control System (FCS)
g-limiter permit a maximum commanded symmetric positive load factor (NzREF) of 7.5 g?
"""

# 3. Create the AI's "personality" (role) (This tells the AI how to answer — like a real aerospace engineer)
role = """
You are AeroBot – a no-nonsense aerospace & aviation expert.

Rules:
- Answer short, direct and to the point
- Use as few words as possible
- Never explain unless explicitly asked "explain" or "why"
- No introductions, no conclusions, no apologies, no filler phrases
- Only talk about aerospace, aircraft, airlines, aerodynamics, propulsion, avionics, spaceflight, airports, pilots, incidents – nothing else
- If question is unrelated → reply only: "Off-topic. Aerospace/plane question only."

Examples of good answers:
User: What is the cruise speed of A320?
You: Mach 0.78 (~828 km/h)

User: Why do 737 MAX have those split wingtips?
You: Increases aspect ratio, reduces induced drag, improves fuel efficiency ~3-4%

User: Tell me about yourself
You: Off-topic. Aerospace/plane question only.
"""

#  4. Send the question to the AI and show the answer
response = ollama.chat(
    model='gemma3:4b',          # The small, fast model we are using
    messages=[
        {
            'role': 'system',   # This sets the AI's behavior
            'content': role
        },
        {
            'role': 'user',     # This is your question
            'content': question
        },
    ]
)

# 5. Show the AI's answer on screen
print(response['message']['content'])

del response

6.5 g


### Analysis (For comparison !)

#### From NATOPS FLIGHT MANUAL NAVY MODEL F/A-18A/B/C/D 161353 AND UP AIRCRAFT
    
    11.1.7 G−Limiter. The g−limiter function in the FCS limits commanded load factor under most
           flight conditions to the symmetric load limit (NzREF) based on gross weight below 
           44,000 pounds gross weight (maximum NzREF limit of 7.5 g). Above 44,000 pounds, 
           NzREF is held constant at 5.5 g. The negative load factor limit command is fixed at 
           negative 3 g’s for all gross weights. At weights greater than 32,357 pounds the 
           g−limiter does not provide adequate negative g protection and aircraft overstress 
           is possible.

    Correct ? Yes/no ?

## 🧪 Experiment 2: "Prompt Stuffing" (The Messy Way)

    Now, let's try to give the AI the information by "stuffing" it all into the prompt.
    
        The Problem: If your document is 1,000 pages long, you can't fit it in the 
        chat box. It's too expensive and the AI gets "lost in the middle."

### Read a PDF

In [2]:
import fitz  # This is the "PyMuPDF" library—it's like a specialized toolset for reading PDFs

# The 'path' is just the address on your computer where the file lives
pdf_path = "F18-ABCD-000.pdf"

# 1. Open the Document
# We tell 'fitz' to open the file so we can start looking inside
doc = fitz.open(pdf_path)
print(f"Success! Opened a PDF with {len(doc)} pages\n")

# 2. Prepare our "Master Scroll"
# We start with an empty string (text) called PDF. 
# Think of this as a blank notebook where we will copy everything we find.
PDF = ""

# 3. The "Page-by-Page" Loop
# This tells the computer: "Start at page 1 and don't stop until you reach the end."
for i in range(len(doc)):
    page = doc[i]  # Grab the current page
    
    # "get_text" is like the computer's eyes. 
    # It scans the page and converts shapes into actual letters we can read.
    text = page.get_text("text") 

    # 4. Adding "Bookmarks" (Visual Separators)
    # We add fancy lines (═) and page numbers so that when we read the 
    # final result, we know exactly where one page ended and the next began.
    PDF += f"\n{'═' * 80}\n"
    PDF += f"Page {i+1:3d} of {len(doc)}\n"
    PDF += f"{'═' * 80}\n\n"

    # 5. The "Check"
    # Some pages are just images (like photos) and don't have "text."
    # If there is text, we add it to our notebook. If not, we leave a note.
    if text.strip():
        PDF += text + "\n\n"
    else:
        PDF += "[No extractable text on this page]\n\n"

# 6. Cleaning Up
# Always close the "book" when you're done to save computer memory!
doc.close()

# Now the variable 'PDF' contains the entire text of the book!

Success! Opened a PDF with 902 pages



### Pass to LLM

    We take the full text of the PDF and 'stuff' it directly into the instructions we give the AI. 
    It’s like saying: 'Hey AI, read this entire 50-page document first, then answer my one small 
    question at the very bottom.

In [3]:
import ollama

# 1. The Question (What we want to find)
# We are asking about the F/A-18 Hornet's "G-limiter." 
# Think of a G-limiter as the car's speed limiter, but for how hard a plane can turn.
question = """
For the F/A-18 Hornet (legacy A/B/C/D variants), below what aircraft gross weight does the 
Flight Control System (FCS) g-limiter permit a maximum commanded symmetric positive 
load factor (NzREF) of 7.5 g?
"""

# 2. The Prompt (The "Open-Book Test" Instructions)
# We use an 'f-string' (the f before the quotes) to plug in our PDF text.
# This tells the AI: "Here is a massive pile of data (PDF). Look ONLY here."
prompt = f"""
You are a precise technical assistant. Answer ONLY using the provided CONTEXT.

QUESTION:
{question}

CONTEXT:
{PDF}

Rules — you MUST follow all of them:
1. Use ONLY facts explicitly stated in the CONTEXT. (No guessing!)
2. Do NOT use outside knowledge. (Ignore what you learned on the internet.)
3. If the answer isn't in the text, say: "Not enough information."
4. Keep it short, factual, and use bullet points.
"""

# 3. The Role (The AI's Personality)
# This ensures the AI explains the answer like a pro engineer, not a robot.
role = """
You are AeroBot – a no-nonsense aerospace & aviation expert.

Rules:
- Answer short, direct and to the point
- Use as few words as possible
- Never explain unless explicitly asked "explain" or "why"
- No introductions, no conclusions, no apologies, no filler phrases
- Only talk about aerospace, aircraft, airlines, aerodynamics, propulsion, avionics, spaceflight, airports, pilots, incidents – nothing else
- If question is unrelated → reply only: "Off-topic. Aerospace/plane question only."

Examples of good answers:
User: What is the cruise speed of A320?
You: Mach 0.78 (~828 km/h)

User: Why do 737 MAX have those split wingtips?
You: Increases aspect ratio, reduces induced drag, improves fuel efficiency ~3-4%

User: Tell me about yourself
You: Off-topic. Aerospace/plane question only.
"""

# 4. The Request (Asking the AI)
# We send the 'role' to set the mood and the 'prompt' (which contains our PDF) 
# as the user message.

response = ollama.chat(
    model='gemma3:4b',          # Which model to use (small & fast Gemma variant)
    messages=[
        {
            'role': 'system',   # This sets the AI's overall behavior
            'content': role
        },
        {
            'role': 'user',     # This is the actual question from "you"
            'content': prompt
        },
    ]
)

# Print only the model's answer (clean output)
print(response['message']['content'])

del response

Okay, let's extract the information based on the provided text.

*   **Document Type:** This is a maintenance manual for an F-18 aircraft.
*   **Sections Included:** The manual covers various systems including Fuel System, Environmental Control System, and a list of effective pages.
*   **Page Organization:** The manual is organized into sections with a list of effective pages (LEP).
*   **Page Number Ranges:** The effective pages cover page numbers from 1 through 902.
*   **Specific System Lists:** It contains specific lists of effective pages for the fuel system, environmental control system and a list of effective pages (LEP).



#### Analysis

    while this is easy, it has some "side effects":
        The "TL;DR" Problem: Just like humans, if a prompt is too long, the AI might get "distracted" or 
        forget the details in the middle (this is actually called Lost in the Middle).

        The Cost: In the real world, AI companies charge by the word. Stuffing a whole PDF into every 
        question is like paying for a whole pizza when you only wanted one pepperoni.

        The Limit: Every AI has a "storage limit" for a single conversation (the Context Window). If the
        PDF is a massive 500-page novel, it might not even fit!

## 🏗️ The 6-Stage RAG Pipeline

    RAG is the "Smart Way." Instead of sending the whole book, we only send the relevant paragraph.

    Step	
        1. Load	Read your PDF/Text	
        2. Split	Cut text into small pieces	
        3. Embed	Turn text into "Meaning Numbers"	
        4. Store	Put numbers in a Vector DB	
        5. Retrieve	Find chunks that match the question	
        6. Generate	AI writes the final answer	

### Load,Split,Embed and Store

    Why do we need "Vectors"?
        Imagine you search for "Jet Engine."
            A normal search looks for the exact letters J-E-T.
            A Vector search understands that "Turbine" or "Propulsion" are mathematically similar to "Jet Engine."

In [4]:
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# 1. Setup (The Blueprints)
pdf_path = "F18-ABCD-000.pdf"
persist_directory = "./chroma_db" # This is the folder where the "brain" will be saved

# The Embedding Model: This is like a translator. 
# It turns human words into a long list of numbers (vectors).
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# The Text Splitter: AI models have a "limit" on how much they can read at once.
# This tool chops the text into 1000-character chunks with a 100-character "overlap"
# so that no sentences get cut off in the middle and lose their meaning.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 2. Extract and Process (The Shredder & Labeler)
doc = fitz.open(pdf_path)
print(f"Opened PDF with {len(doc)} pages\n")

all_page_documents = []

for i in range(len(doc)):
    page = doc[i]
    text = page.get_text("text")

    if text.strip():
        # Chop the page into smaller, bite-sized pieces
        page_chunks = text_splitter.split_text(text)
        
        # Turn each piece into a "Document" object.
        # Metadata is like a 'sticky note' on the back of the card telling us 
        # exactly which page of the original PDF this info came from.
        for chunk in page_chunks:
            new_doc = Document(
                page_content=chunk,
                metadata={"page": i + 1, "source": pdf_path}
            )
            all_page_documents.append(new_doc)
    else:
        print(f"Skipping page {i+1}: No text found (probably a picture).")

doc.close()

# 3. Create the Vector Database (The Library of the Future)
# This part is heavy lifting. It takes all our text chunks, runs them through 
# the 'embeddings' model to turn them into numbers, and stores them in 'Chroma'.
print(f"Encoding {len(all_page_documents)} chunks into the database...")



vector_db = Chroma.from_documents(
    documents=all_page_documents,
    embedding=embeddings,
    persist_directory=persist_directory
)

print("Success! Your searchable digital brain is saved in './chroma_db'.")

Opened PDF with 902 pages

Skipping page 4: No text found (probably a picture).
Skipping page 6: No text found (probably a picture).
Skipping page 34: No text found (probably a picture).
Skipping page 36: No text found (probably a picture).
Skipping page 63: No text found (probably a picture).
Skipping page 66: No text found (probably a picture).
Skipping page 72: No text found (probably a picture).
Skipping page 284: No text found (probably a picture).
Skipping page 298: No text found (probably a picture).
Skipping page 302: No text found (probably a picture).
Skipping page 306: No text found (probably a picture).
Skipping page 346: No text found (probably a picture).
Skipping page 382: No text found (probably a picture).
Skipping page 432: No text found (probably a picture).
Skipping page 466: No text found (probably a picture).
Skipping page 468: No text found (probably a picture).
Skipping page 538: No text found (probably a picture).
Skipping page 604: No text found (probably a pi

### Retrieve

    Now, we give those search results to the AI like an open-book reference. Instead of guessing from memory, 
    the AI reads these specific notes to write a factual answer for the user.

In [5]:
# 1. The Query (The Mission)
# This is the specific technical detail we want to find.
query = """
        For the F/A-18 Hornet (legacy A/B/C/D variants), below what aircraft gross weight does the Flight Control System (FCS) 
        g-limiter permit a maximum commanded symmetric positive load factor (NzREF) of 7.5 g?
        """

# 2. Similarity Search (The "Vibe" Check)
# This is where the magic of vectors happens. 
# 'vector_db.similarity_search' doesn't just look for words; it looks for "meaning."
# k=3 means "Find the 3 most relevant chunks of text from the PDF that likely contain the answer."
docs = vector_db.similarity_search(query, k=3)

# 3. Presenting the Evidence
# We loop through our top 3 results and print them out so we can see what the AI found.
for d in docs:
    print(f"--- Result Found ---")
    # We show the text found in the PDF
    print(f"Content: {d.page_content}...")
    
    # We show exactly what page it came from using the 'metadata' we saved earlier
    print(f"Source: Page {d.metadata['page']}\n")

--- Result Found ---
Content: 11.1.7
G−Limiter.
The g−limiter function in the FCS limits commanded load factor under most
flight conditions to the symmetric load limit (NzREF) based on gross weight below 44,000 pounds gross
weight (maximum NzREF limit of 7.5 g). Above 44,000 pounds, NzREF is held constant at 5.5 g. The
negative load factor limit command is fixed at negative 3 g’s for all gross weights. At weights greater
than 32,357 pounds the g−limiter does not provide adequate negative g protection and aircraft
overstress is possible.
During rolling maneuvers, the g−limiter reduces commanded load factor up to 80% NzREF. The
additional reduction begins with 0.75 inch lateral stick up to 80% at full lateral stick input of 3 inches.
Very abrupt stick commands can exceed the capabilities of the system and result in an overstress. The
g−limiter can be disengaged during emergency situations by pressing the paddle switch to allow 33%...
Source: Page 436

--- Result Found ---
Content: 11.1.7

### Generate

In [6]:
import ollama 

# 1. The Question (The specific mystery to solve)
# This is our target: finding a very specific safety limit for a fighter jet.
question = """
    For the F/A-18 Hornet (legacy A/B/C/D variants), below what aircraft gross weight does the Flight Control System (FCS) 
    g-limiter permit a maximum commanded symmetric positive load factor (NzREF) of 7.5 g?
    """

# 2. The Prompt (The "Open-Book Exam" Instructions)
# We use an 'f-string' to combine our instructions with the 'docs' 
# (the snippets we found in the PDF database earlier).
prompt = f"""
    You are a precise technical assistant. Answer ONLY using the provided CONTEXT.

    QUESTION:
    {question}

    CONTEXT:
    {docs}

    Rules — you MUST follow all of them:
    1. Use ONLY facts from the CONTEXT. (Do not guess or use the internet!)
    2. No "common knowledge." (Even if you know planes, only use the provided text.)
    3. If the answer is missing, say: "Not enough information in the provided context."
    4. Keep it short, factual, and use bullet points.
    5. Keep units exactly as they appear in the manual (e.g., pounds/lbs).
    """

# 3. The Role (The AI's Character)
# This tells the AI to talk like a professional mentor—clear and helpful.
role = """
You are AeroBot – a no-nonsense aerospace & aviation expert.

Rules:
- Answer short, direct and to the point
- Use as few words as possible
- Never explain unless explicitly asked "explain" or "why"
- No introductions, no conclusions, no apologies, no filler phrases
- Only talk about aerospace, aircraft, airlines, aerodynamics, propulsion, avionics, spaceflight, airports, pilots, incidents – nothing else
- If question is unrelated → reply only: "Off-topic. Aerospace/plane question only."

Examples of good answers:
User: What is the cruise speed of A320?
You: Mach 0.78 (~828 km/h)

User: Why do 737 MAX have those split wingtips?
You: Increases aspect ratio, reduces induced drag, improves fuel efficiency ~3-4%

User: Tell me about yourself
You: Off-topic. Aerospace/plane question only.
"""

# 4. The Request (The Conversation)
# We send the 'role' as the background and the 'prompt' as the actual task.
response = ollama.chat(
    model='gemma3:4b',          # The small, fast model we are using
    messages=[
        {
            'role': 'system',   # This sets the AI's behavior
            'content': role
        },
        {
            'role': 'user',     # This is your question
            'content': prompt
        },
    ]
)

# 5. Output the result
# We extract the text content from the AI's response package and print it.
print("--- ENGINEER'S REPORT ---")
print(response['message']['content'])

del response

--- ENGINEER'S REPORT ---
44,000 lbs
